In [6]:
import kagglehub

In [7]:
import torch
import torch.nn as nn
from torchvision import models,datasets,transforms
from torch.utils.data import DataLoader
import torch.optim as optim

In [8]:
path = kagglehub.dataset_download("tongpython/cat-and-dog")

Using Colab cache for faster access to the 'cat-and-dog' dataset.


In [9]:
print(path)

/kaggle/input/cat-and-dog


In [11]:
import os
os.listdir("/kaggle/input/cat-and-dog")


['test_set', 'training_set']

In [12]:
os.listdir("/kaggle/input/cat-and-dog/training_set")


['training_set']

In [13]:
os.listdir("/kaggle/input/cat-and-dog/training_set")


['training_set']

In [15]:
os.listdir("/kaggle/input/cat-and-dog/training_set/training_set")


['dogs', 'cats']

In [16]:
TRAIN_DIR = "/kaggle/input/cat-and-dog/training_set/training_set"
TEST_DIR  = "/kaggle/input/cat-and-dog/test_set/test_set"


In [17]:
if torch.cuda.is_available():
    device=torch.device("cuda")
else:
    device=torch.device("cpu")
model=models.resnet18(pretrained=True)
criterion=nn.CrossEntropyLoss()
model.fc=nn.Linear(model.fc.in_features,2)
model=model.to(device)
for param in model.parameters():
    param.requires_grad=False
for param in model.layer4.parameters():
    param.requires_grad=True
for param in model.layer3.parameters():
    param.requires_grad=True
for param in model.fc.parameters():
    param.requires_grad=True
optimize=optim.Adam([{"params":model.layer3.parameters(),"lr":1e-4},{'params':model.layer4.parameters(),'lr':1e-4},{'params':model.fc.parameters(),'lr':1e-3}])
scheduler = optim . lr_scheduler . StepLR ( optimize, step_size =10 , gamma
=0.1)
def dataset_loading():
    train_transform=transforms.Compose([transforms.RandomResizedCrop(224),transforms.RandomHorizontalFlip(),transforms.ToTensor(),transforms.Normalize(mean=[0.485,0.456,0.406],std=[0.229,0.224,0.225])])
    test_transform=transforms.Compose([transforms.Resize((224, 224)),transforms.ToTensor(),transforms.Normalize(mean=[0.485,0.456,0.406],std=[0.229,0.224,0.225])])
    train_set=datasets.ImageFolder(TRAIN_DIR, transform=train_transform)
    test_set=datasets.ImageFolder(TEST_DIR, transform=test_transform)
    train_loader=DataLoader(train_set,shuffle=True,batch_size=32)
    test_loader=DataLoader(test_set,shuffle=False,batch_size=32)
    return train_loader,test_loader
def train_data(train_loader,model):

    model.train()
    for epoch in range(10):
        loss_per_epoch=0
        total=0
        for images,labels in train_loader:
            images=images.to(device)
            labels=labels.to(device)
            output=model(images)
            loss=criterion(output,labels)
            optimize.zero_grad()
            loss.backward()
            optimize.step()
            loss_per_epoch+=loss.item()

        scheduler.step()
        print(f"Epoch {epoch+1},Loss:{loss_per_epoch/len(train_loader)}")
def test_data(test_loader,model):
  model.eval()
  correct=0
  test_total=0

  with torch.no_grad():
        for images,labels in test_loader:
            images=images.to(device)
            labels=labels.to(device)
            test_total+=labels.size(0)
            output=model(images)
            _,predicted=torch.max(output,1)
            correct+=(predicted==labels).sum().item()



        accuracy=correct/test_total
        print(f"Test Accuracy with test_set:{accuracy*100}%")
def main():
    train_loader,test_loader=dataset_loading()
    train_data(train_loader,model)
    test_data(test_loader,model)
main()




/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 170MB/s]


Epoch 1,Loss:0.1726706768144887
Epoch 2,Loss:0.13112755551325728
Epoch 3,Loss:0.11369896698297852
Epoch 4,Loss:0.10804310672178093
Epoch 5,Loss:0.09636374198241657
Epoch 6,Loss:0.0891909738891272
Epoch 7,Loss:0.08375961102503054
Epoch 8,Loss:0.08427434038803115
Epoch 9,Loss:0.08943982082103709
Epoch 10,Loss:0.09611443250970655
Test Accuracy with test_set:98.91250617894217%


In [18]:
torch.save(model.state_dict(), "dogs_vs_cats.pth")

In [19]:
from google.colab import files
files.download("dogs_vs_cats.pth")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>